# 第 45 课：迷你音频语言模型——Audio Encoder 如何接入 LLM

现代 Large Audio-Language Model（LALM）不再把语音只看作逐帧分类问题，而是把连续音频表示接入语言模型，让语言模型自回归生成转录。

本课在 CPU 上训练一个完整的迷你系统：`Audio Encoder → Projector → Decoder-only LM → 文本 token`，并实现 teacher forcing 与 greedy generation。

## 完成标准

1. 解释 Audio Encoder、Projector、LLM 各自职责；
2. 写出音频帧率下采样前后的 shape；
3. 解释为什么音频 prefix 不计算 next-token loss；
4. 训练迷你 LALM 并用未见样本自回归生成；
5. 说明生成式 ASR 的上下文优势与幻觉风险。

本课是结构复现，不是 Qwen3-ASR 等基础模型的规模复现。

## 课前诊断

1. 如果声学 encoder 每秒输出 100 帧，而 LLM 每秒只接受约 10 个音频 embedding，需要哪个模块缩短序列？
2. LLM 生成第 5 个文字 token 时可以依赖什么？
3. 为什么语言能力越强，ASR 不一定越忠于原始声音？

<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC/流式状态和延迟；训练/测试数据权限；基线、消融与外部评测证据。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](44_自监督语音预训练_遮挡与声学表示.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：明确任务、数据、算力、延迟与风险约束
  ↓ 本课要学会的变换、状态或判断
输出：能与 CTC/RNN-T/AED/LALM 基线公平比较的现代模型实验
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [1]:
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(31)
np.random.seed(31)
random.seed(31)
torch.set_num_threads(2)
device = torch.device("cpu")
print("torch:", torch.__version__, "device:", device)

torch: 2.13.0+cpu device: cpu


## 1. 架构与 shape

```text
音频 [B,T,F]
  → Audio Encoder（局部特征 + 下采样）
  → speech embeddings [B,T',D_a]
  → Projector
  → audio prefix [B,T',D_lm]
  → 与 [BOS, y1, y2, ...] 的文字 embedding 拼接
  → Decoder-only Transformer
  → next-token logits
```

Projector 不只是“改 shape”。实际系统中，它还要完成两种表示空间的对齐，并控制音频序列长度，否则 LLM 的注意力成本会过高。

Qwen3-ASR 的公开报告采用 AuT encoder、8 倍下采样、projector 和 Qwen3 LLM；本课用更小的卷积 encoder 和 Transformer 复现相同接口思想。

## 2. 合成语音—文字对

设 8 个内容 token，每个 token 对应一个带噪声声学原型并持续 3 帧。文字词表另外包含 `PAD/BOS/EOS`。

与第 44 课不同，本课训练目标是文字 next-token，而不是离散声学 code。

In [2]:
PAD, BOS, EOS = 0, 1, 2
NUM_CONTENT = 8
VOCAB_SIZE = 3 + NUM_CONTENT
FEAT_DIM = 16
TOKENS_PER_UTT = 6
FRAMES_PER_TOKEN = 3
AUDIO_LEN = TOKENS_PER_UTT * FRAMES_PER_TOKEN

data_generator = torch.Generator().manual_seed(2026)
audio_prototypes = F.normalize(
    torch.randn(NUM_CONTENT, FEAT_DIM, generator=data_generator), dim=-1
)


def make_audio_text_batch(batch_size, generator=None, noise_std=0.10):
    generator = generator or torch.default_generator
    content = torch.randint(
        0, NUM_CONTENT, (batch_size, TOKENS_PER_UTT), generator=generator
    )
    text = content + 3
    frame_codes = content.repeat_interleave(FRAMES_PER_TOKEN, dim=1)
    noise = noise_std * torch.randn(
        batch_size, AUDIO_LEN, FEAT_DIM, generator=generator
    )
    audio = audio_prototypes[frame_codes] + noise
    return audio, text


audio, text = make_audio_text_batch(2, torch.Generator().manual_seed(5))
print("audio:", tuple(audio.shape), "text:", tuple(text.shape))
print("sample text ids:", text[0].tolist())
assert audio.shape == (2, 18, 16) and text.shape == (2, 6)
print("断言通过：每个文字 token 对应 3 个带噪声声学帧。")

audio: (2, 18, 16) text: (2, 6)
sample text ids: [6, 9, 10, 8, 9, 9]
断言通过：每个文字 token 对应 3 个带噪声声学帧。


## 3. 搭建迷你 LALM

卷积 `kernel_size=stride=3` 把 18 帧压到 6 个 speech embedding。真实系统通常使用多层卷积、Conformer/Transformer 和更复杂的下采样。

语言模型输入序列为：

```text
[audio_1 ... audio_6] [BOS, y1, y2, ... y6]
```

文字位置的监督目标为：

```text
[y1, y2, ... y6, EOS]
```

因此只在最后 7 个文字位置计算交叉熵；音频 prefix 提供条件，不要求它预测文字。

In [3]:
class MiniAudioEncoder(nn.Module):
    def __init__(self, feat_dim=FEAT_DIM, audio_dim=32):
        super().__init__()
        self.subsample = nn.Conv1d(
            feat_dim, audio_dim,
            kernel_size=FRAMES_PER_TOKEN,
            stride=FRAMES_PER_TOKEN,
        )
        self.norm = nn.LayerNorm(audio_dim)

    def forward(self, audio):
        x = self.subsample(audio.transpose(1, 2)).transpose(1, 2)
        return self.norm(F.silu(x))


class MiniAudioLanguageModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, audio_dim=32, d_model=48):
        super().__init__()
        self.audio_encoder = MiniAudioEncoder(audio_dim=audio_dim)
        self.projector = nn.Sequential(
            nn.Linear(audio_dim, d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position = nn.Parameter(torch.randn(1, 32, d_model) * 0.01)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=128,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.lm = nn.TransformerEncoder(layer, num_layers=2)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def audio_prefix(self, audio):
        return self.projector(self.audio_encoder(audio))

    def forward(self, audio, text_input):
        prefix = self.audio_prefix(audio)
        text_emb = self.token_embedding(text_input)
        x = torch.cat([prefix, text_emb], dim=1)
        x = x + self.position[:, :x.size(1)]
        causal = torch.triu(
            torch.ones(x.size(1), x.size(1), dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        hidden = self.lm(x, mask=causal, is_causal=True)
        return self.lm_head(hidden), prefix.size(1)


lal_model = MiniAudioLanguageModel().to(device)
text_input = torch.cat([
    torch.full((2, 1), BOS, dtype=torch.long), text
], dim=1)
all_logits, prefix_len = lal_model(audio, text_input)
text_logits = all_logits[:, prefix_len:]
print("audio prefix length:", prefix_len)
print("all logits:", tuple(all_logits.shape))
print("supervised text logits:", tuple(text_logits.shape))
assert prefix_len == 6
assert text_logits.shape == (2, 7, VOCAB_SIZE)
print("断言通过：18 个声学帧 → 6 个 prefix；7 个文字位置接受监督。")

audio prefix length: 6
all logits: (2, 13, 11)
supervised text logits: (2, 7, 11)
断言通过：18 个声学帧 → 6 个 prefix；7 个文字位置接受监督。


<USER_HOME>\AppData\Local\Temp\ipykernel_30688\86677357.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.lm = nn.TransformerEncoder(layer, num_layers=2)


## 4. Teacher forcing 训练

训练时将正确历史 `[BOS,y1,...]` 输入模型，这称为 teacher forcing。模型学习在音频和正确文字前缀条件下预测下一个 token。

推理时没有正确未来文字，只能把自己的输出逐步送回模型，因此错误可能累积。

In [4]:
def lal_loss(model, audio, target_text):
    bos = torch.full(
        (target_text.size(0), 1), BOS,
        dtype=torch.long, device=target_text.device,
    )
    text_input = torch.cat([bos, target_text], dim=1)
    text_target = torch.cat([
        target_text,
        torch.full_like(bos, EOS),
    ], dim=1)
    logits, prefix_len = model(audio, text_input)
    supervised_logits = logits[:, prefix_len:]
    loss = F.cross_entropy(
        supervised_logits.reshape(-1, VOCAB_SIZE),
        text_target.reshape(-1),
    )
    accuracy = (
        supervised_logits.argmax(-1) == text_target
    ).float().mean()
    return loss, accuracy


optimizer = torch.optim.AdamW(lal_model.parameters(), lr=3e-3)
train_gen = torch.Generator().manual_seed(77)
history = []
lal_model.train()
for step in range(181):
    batch_audio, batch_text = make_audio_text_batch(48, train_gen)
    loss, accuracy = lal_loss(lal_model, batch_audio, batch_text)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(lal_model.parameters(), 5.0)
    optimizer.step()
    if step % 30 == 0:
        history.append((step, loss.item(), accuracy.item()))

for step, loss_value, acc_value in history:
    print(f"step={step:3d} loss={loss_value:.4f} teacher_forced_acc={acc_value:.3f}")

assert history[-1][1] < history[0][1] * 0.35
print("断言通过：next-token loss 显著下降。")

step=  0 loss=2.6201 teacher_forced_acc=0.086
step= 30 loss=1.1558 teacher_forced_acc=0.551
step= 60 loss=0.6623 teacher_forced_acc=0.750
step= 90 loss=0.3231 teacher_forced_acc=0.869
step=120 loss=0.1603 teacher_forced_acc=0.940
step=150 loss=0.1405 teacher_forced_acc=0.949
step=180 loss=0.0403 teacher_forced_acc=0.985
断言通过：next-token loss 显著下降。


## 5. 真正的自回归生成

下面不提供正确转录。模型从 `[BOS]` 开始，每次取最后一个位置的最大概率 token，追加后重新运行，直到生成 `EOS` 或达到上限。

In [5]:
@torch.no_grad()
def greedy_generate(model, audio, max_new_tokens=10):
    model.eval()
    history = torch.full(
        (audio.size(0), 1), BOS, dtype=torch.long, device=audio.device
    )
    finished = torch.zeros(audio.size(0), dtype=torch.bool, device=audio.device)
    outputs = [[] for _ in range(audio.size(0))]

    for _ in range(max_new_tokens):
        logits, _ = model(audio, history)
        next_token = logits[:, -1].argmax(-1)
        history = torch.cat([history, next_token.unsqueeze(1)], dim=1)
        for b, token in enumerate(next_token.tolist()):
            if not finished[b]:
                if token == EOS:
                    finished[b] = True
                else:
                    outputs[b].append(token)
        if finished.all():
            break
    return outputs


test_audio, test_text = make_audio_text_batch(
    64, torch.Generator().manual_seed(12345)
)
generated = greedy_generate(lal_model, test_audio)
token_correct = 0
token_total = 0
exact = 0
for prediction, reference in zip(generated, test_text.tolist()):
    token_correct += sum(a == b for a, b in zip(prediction, reference))
    token_total += max(len(reference), len(prediction))
    exact += int(prediction == reference)

print("前 5 条未见样本：")
for i in range(5):
    print("reference=", test_text[i].tolist(), "generated=", generated[i])
print(f"token accuracy={token_correct/token_total:.3f}")
print(f"exact sequence accuracy={exact/len(generated):.3f}")
assert token_correct / token_total > 0.90
assert exact / len(generated) > 0.65
print("断言通过：模型在未见音频上完成了自回归转录。")

前 5 条未见样本：
reference= [5, 8, 8, 4, 7, 4] generated= [5, 8, 8, 4, 7, 4]
reference= [9, 9, 8, 5, 8, 4] generated= [9, 9, 8, 5, 8, 4]
reference= [9, 4, 6, 9, 4, 6] generated= [9, 4, 6, 9, 6, 4]
reference= [10, 9, 3, 5, 4, 6] generated= [10, 9, 3, 5, 4, 6]
reference= [8, 5, 4, 5, 9, 10] generated= [8, 5, 4, 5, 9, 10]
token accuracy=0.990
exact sequence accuracy=0.969
断言通过：模型在未见音频上完成了自回归转录。


## 6. 为什么 LALM 强，也为什么会幻觉？

语言模型可以利用：

- 已生成文字、对话历史、领域词表和世界知识；
- 多语言共用表示；
- 指令或上下文中的专名提示。

但 next-token 目标优化的是“条件下最可能的文字”，不保证每个字都有充分声学证据。当输入是静音、强噪声或分布外声音时，强语言先验可能生成流畅但错误的内容。

生产系统至少要做：

1. 静音/非语音负样本训练；
2. 空转录能力与 VAD；
3. 数字、姓名、否定词的证据保护；
4. 噪声、截断、音乐和提示注入测试；
5. 保存原始转录、置信度和审计信息。

In [6]:
# 反事实测试：把音频替换为全零，观察模型是否仍生成内容。
silent_audio = torch.zeros_like(test_audio[:8])
silent_outputs = greedy_generate(lal_model, silent_audio)
print("全零输入的生成结果：")
for output in silent_outputs:
    print(output)

nonempty = sum(bool(x) for x in silent_outputs)
print(f"非空输出 {nonempty}/{len(silent_outputs)}")
print("这不是准确率测试，而是生成式 ASR 必须具备的静音安全测试。")

全零输入的生成结果：
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
[5, 6, 7, 6, 9, 7]
非空输出 8/8
这不是准确率测试，而是生成式 ASR 必须具备的静音安全测试。


## 7. 从本课到 Qwen3-ASR

| 本课 | 大规模 LALM |
|---|---|
| 18 帧合成特征 | 真实 Fbank/波形与长音频 |
| 单层卷积 encoder | 数亿参数音频 encoder |
| 小 projector | 多层对齐与下采样模块 |
| 2 层 Transformer | 预训练 LLM |
| 180 步随机数据 | 海量预训练、SFT、RL |

搭建现实系统的合理方式不是从零复现数千万小时训练，而是：理解并验证本课接口 → 选择开放 checkpoint → 准备合法领域数据 → 参数高效或全量微调 → 做独立真实评测。

## 分层练习（24 分）

### A. 回忆（每题 1 分）

1. Audio Encoder、Projector、LLM 各做什么？
2. 为什么 projector 常伴随下采样？
3. teacher forcing 和推理的输入历史有何不同？
4. 本课哪些位置计算交叉熵？

### B. 推理（每题 2 分）

5. 输入 1,000 帧、下采样 8 倍，prefix 约多长？
6. prefix 长度翻倍时，普通全注意力计算量约变为多少倍？
7. 为什么静音也生成文字属于严重失败？
8. LALM 上下文 biasing 与传统 WFST 热词各有什么风险？

### C. 编程（每题 3 分）

9. 将 noise_std 提高到 0.5，重新评估生成准确率。
10. 删除 projector 的非线性层，比较收敛速度。
11. 增加 `NO_SPEECH` 样本，使静音输出 EOS。
12. 从空白实现 greedy generation，禁止读取参考文字。

达到 19/24 且通过“生成不读取 reference”的代码审查，再进入真实 Qwen3-ASR。

## 离场小测（闭卷发给老师）

1. 用自己的话解释“把音频作为 LLM prefix”。
2. 写出 `[B,T,F] → [B,T',D_lm] → token` 的 shape 链。
3. teacher forcing 准确率高为什么不能证明真实生成准确率高？
4. 列出三项生成式 ASR 的幻觉测试。

附上你的 token/exact accuracy、静音输出和最没有把握的一题。